# Power Quality EDA
Exploratory analysis of active/reactive power, accumulated energy, power factor, peak behavior, and module-level operating patterns.


## **데이터 불러오기**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

In [ ]:
import pandas as pd

df1 = pd.read_csv('rtu_2025_02.csv')
df = df1.copy()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df['localtime'] = pd.to_datetime(
    df['localtime'].astype(str),
    format='%Y%m%d%H%M%S'
)

df['localtime'].head()

In [ ]:
modules = df['module(equipment)'].unique()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

numeric_df = df.select_dtypes(include=[np.number])

corr = numeric_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, 
            annot=True,
            fmt=".2f",
            cmap='RdYlGn',
            center=0,
            linewidths=0.5)
plt.title("Correlation Heatmap of All Features", fontsize=15)
plt.show()

## 장비(module(equipment))별 EDA

데이터가 너무 많아서 100개당 1개로 샘플링하여 진행

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

STEP = 100

# 1. 시간별 주요 지표 추이 (ActivePower, Lagging)
def plot_time_series_pattern(mod, m_df):
    plt.figure(figsize=(15, 6))
    plt.plot(m_df['localtime'][::STEP], m_df['activePower'][::STEP], label='ActivePower', alpha=0.8)
    plt.plot(m_df['localtime'][::STEP], m_df['reactivePowerLagging'][::STEP], label='Lagging', alpha=0.6)
    plt.title(f"[{mod}] Time-series Power Patterns (Sampling 1/{STEP})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 2. ActivePower 중심 상관관계 히트맵
def plot_active_power_corr(mod, m_df):
    plt.figure(figsize=(10, 8))
    numeric_cols = m_df.select_dtypes(include=[np.number]).columns
    corr = m_df[numeric_cols].corr()
    sns.heatmap(corr[['activePower']].sort_values(by='activePower', ascending=False), 
                annot=True, cmap='RdYlGn', center=0)
    plt.title(f"[{mod}] Correlation with ActivePower")
    plt.show()

# 3. 무효전력(Lagging) 및 효율 지표 분석
def plot_lagging_efficiency(mod, m_df):
    m_df['avg_PF'] = m_df[['powerFactorR', 'powerFactorS', 'powerFactorT']].mean(axis=1)
    
    fig, ax1 = plt.subplots(figsize=(15, 6))
    ax2 = ax1.twinx()
    
    ax1.fill_between(m_df['localtime'][::STEP], m_df['reactivePowerLagging'][::STEP], 
                     color='purple', alpha=0.3, label='Lagging')
    ax2.plot(m_df['localtime'][::STEP], m_df['avg_PF'][::STEP], 
             color='red', label='Avg PowerFactor')
    
    # 정규화된 유효전력
    norm_active = m_df['activePower'] / m_df['activePower'].max()
    ax2.plot(m_df['localtime'][::STEP], norm_active[::STEP], 
             color='green', label='Normalized ActivePower', alpha=0.5)
    
    ax1.set_ylabel('Reactive Power Lagging')
    ax2.set_ylabel('PF / Norm ActivePower')
    plt.title(f"[{mod}] Lagging vs Efficiency Indicators (Sampling 1/{STEP})")
    ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
    plt.grid(True, alpha=0.2)
    plt.show()

# 4. 에너지 사용량 변화량(Energy Usage) 확인
def plot_energy_usage_diff(mod, m_df):
    # 변화량 계산은 전체 데이터에서 한 뒤 시각화 시에 샘플링
    m_df['energy_usage'] = m_df['accumActiveEnergy'].diff().fillna(0)
    m_df.loc[m_df['energy_usage'] < 0, 'energy_usage'] = 0
    
    plt.figure(figsize=(15, 5))
    # bar 차트는 데이터가 너무 많으면 깨지므로 선 그래프나 fill_between 추천 (혹은 width 조절)
    plt.plot(m_df['localtime'][::STEP], m_df['energy_usage'][::STEP], color='orange', label='Energy Usage (Diff)')
    plt.fill_between(m_df['localtime'][::STEP], m_df['energy_usage'][::STEP], color='orange', alpha=0.3)
    plt.title(f"[{mod}] Actual Energy Usage per Time Step (Sampling 1/{STEP})")
    plt.ylabel("Energy Diff")
    plt.legend()
    plt.show()

# 5. R, S, T 세 개의 상 변화 확인
def plot_phase_analysis(mod, m_df):    
    fig, axes = plt.subplots(4, 1, figsize=(15, 22), sharex=True)
    
    # 1. Power Factor (R, S, T) - 역률 균형 확인
    axes[0].plot(m_df['localtime'][::STEP], m_df['powerFactorR'][::STEP], label='PF_R', alpha=0.7)
    axes[0].plot(m_df['localtime'][::STEP], m_df['powerFactorS'][::STEP], label='PF_S', alpha=0.7)
    axes[0].plot(m_df['localtime'][::STEP], m_df['powerFactorT'][::STEP], label='PF_T', alpha=0.7)
    axes[0].set_title(f"[{mod}] 3-Phase Power Factor (%)", fontsize=14)
    axes[0].set_ylabel("Percentage (%)")
    axes[0].legend(loc='upper right')
    axes[0].grid(True, alpha=0.3)

    # 2. Current (R, S, T) - 부하 균형 확인
    axes[1].plot(m_df['localtime'][::STEP], m_df['currentR'][::STEP], label='Curr_R', color='tab:blue')
    axes[1].plot(m_df['localtime'][::STEP], m_df['currentS'][::STEP], label='Curr_S', color='tab:orange')
    axes[1].plot(m_df['localtime'][::STEP], m_df['currentT'][::STEP], label='Curr_T', color='tab:green')
    axes[1].set_title(f"[{mod}] 3-Phase Current (A)", fontsize=14)
    axes[1].set_ylabel("Current (A)")
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)

    # 3. Phase Voltage (R, S, T) - 상전압 안정성 확인
    axes[2].plot(m_df['localtime'][::STEP], m_df['voltageR'][::STEP], label='V_R')
    axes[2].plot(m_df['localtime'][::STEP], m_df['voltageS'][::STEP], label='V_S')
    axes[2].plot(m_df['localtime'][::STEP], m_df['voltageT'][::STEP], label='V_T')
    axes[2].set_title(f"[{mod}] 3-Phase Phase Voltage (V)", fontsize=14)
    axes[2].set_ylabel("Voltage (V)")
    axes[2].legend(loc='upper right')
    axes[2].grid(True, alpha=0.3)
    
    # 4. Line Voltage (RS, ST, TR) - 선간전압(실제 설비 전압) 확인
    axes[3].plot(m_df['localtime'][::STEP], m_df['voltageRS'][::STEP], label='V_RS', linestyle='--')
    axes[3].plot(m_df['localtime'][::STEP], m_df['voltageST'][::STEP], label='V_ST', linestyle='--')
    axes[3].plot(m_df['localtime'][::STEP], m_df['voltageTR'][::STEP], label='V_TR', linestyle='--')
    axes[3].set_title(f"[{mod}] Line-to-Line Voltage (RS, ST, TR)", fontsize=14)
    axes[3].set_ylabel("Voltage (V)")
    axes[3].set_xlabel("Time")
    axes[3].legend(loc='upper right')
    axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def run_full_eda(mod, df):
    print(f"장비 {mod}")
    m_df = df[df['module(equipment)'] == mod].copy()
    
    if m_df.empty:
        print("데이터가 없습니다.")
        return

    plot_time_series_pattern(mod, m_df)
    plot_active_power_corr(mod, m_df)
    plot_lagging_efficiency(mod, m_df)
    plot_energy_usage_diff(mod, m_df)
    plot_phase_analysis(mod, m_df)

### 요일별 평균 부하 확인

In [ ]:
def plot_weekday_load_analysis(df):
    # 1. 요일 정보 추출 (0: 월요일, 6: 일요일)
    df['weekday'] = df['localtime'].dt.weekday
    df['day_name'] = df['localtime'].dt.day_name()
    
    # 요일 순서 정렬을 위한 리스트
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    # 2. 요일별 평균 계산
    weekday_avg = df.groupby('day_name')[['activePower', 'reactivePowerLagging']].mean().reindex(weekday_order)
    
    # 3. 시각화
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # 유효전력 막대 그래프
    ax1.bar(weekday_avg.index, weekday_avg['activePower'], color='dodgerblue', alpha=0.7, label='Avg ActivePower')
    ax1.set_ylabel('Active Power (Average)', color='dodgerblue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='dodgerblue')
    
    # 무효전력 선 그래프 (이중 축)
    ax2 = ax1.twinx()
    ax2.plot(weekday_avg.index, weekday_avg['reactivePowerLagging'], color='orchid', marker='o', linewidth=3, label='Avg Lagging')
    ax2.set_ylabel('Reactive Power Lagging (Average)', color='orchid', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='orchid')
    
    plt.title('Average Power Load by Weekday (Feb 2025)', fontsize=15)
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.grid(axis='y', alpha=0.3)
    plt.show()
# 
plot_weekday_load_analysis(df)

1. 유효전력(ActivePower)
- 요일에 관계없이 평균 유효전력(ActivePower)이 약 3000 수준으로 거의 일정하게 유지
> 해당 설비가 공장의 기본 운영을 책임지는 기저 부하(Base Load) 설비임을 증명

2. 무효전력(ReactiveRower Lagging)
- 목요일과 일요일에 무효전력 평균이 상대적으로 높게 나타남
> 같은 일을 하더라도 목요일과 일요일에 전력 효율(역률)이 미세하게 떨어진다는 뜻

### 요일별 시간대별 히트맵

In [ ]:
# 요일별/시간대별 유효전력 히트맵
def plot_weekday_hour_load_analysis(df):
    # 1. 요일 정보 추출 (0: 월요일, 6: 일요일)
    df['weekday'] = df['localtime'].dt.weekday
    df['day_name'] = df['localtime'].dt.day_name()
    
    # 요일 순서 정렬을 위한 리스트
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    # 2. 요일별 평균 계산
    weekday_avg = df.groupby('day_name')[['activePower', 'reactivePowerLagging']].mean().reindex(weekday_order)
    df['hour'] = df['localtime'].dt.hour
    pivot_table = df.pivot_table(index='day_name', columns='hour', values='activePower', aggfunc='mean').reindex(weekday_order)

    plt.figure(figsize=(15, 8))
    sns.heatmap(pivot_table, cmap='YlGnBu', annot=False)
    plt.title('Hourly Load Pattern by Weekday')
    plt.show()

plot_weekday_hour_load_analysis(df)

### 1(PM-3)

#### 1주차

In [ ]:
start_date1 = '2025-02-01'
end_date1 = '2025-02-07'
df_weekly1 = df[(df['localtime'] >= start_date1) & (df['localtime'] < end_date1)].copy()

In [ ]:
run_full_eda(modules[0], df_weekly1)

#### 2주차

In [ ]:
start_date2 = '2025-02-08'
end_date2 = '2025-02-14'
df_weekly2 = df[(df['localtime'] >= start_date2) & (df['localtime'] < end_date2)].copy()

In [ ]:
run_full_eda(modules[0], df_weekly2)

#### 3주차

In [ ]:
start_date3 = '2025-02-15'
end_date3 = '2025-02-21'
df_weekly3 = df[(df['localtime'] >= start_date3) & (df['localtime'] < end_date3)].copy()

In [ ]:
run_full_eda(modules[0], df_weekly3)

#### 4주차

In [ ]:
start_date4 = '2025-02-22'
end_date4 = '2025-02-28'
df_weekly4 = df[(df['localtime'] >= start_date4) & (df['localtime'] < end_date4)].copy()

In [ ]:
run_full_eda(modules[0], df_weekly4)

### 요일별 평균 부하 확인

In [ ]:
def plot_weekday_load_analysis(df):
    # 1. 요일 정보 추출 (0: 월요일, 6: 일요일)
    df['weekday'] = df['localtime'].dt.weekday
    df['day_name'] = df['localtime'].dt.day_name()
    
    # 요일 순서 정렬을 위한 리스트
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    # 2. 요일별 평균 계산
    weekday_avg = df.groupby('day_name')[['activePower', 'reactivePowerLagging']].mean().reindex(weekday_order)
    
    # 3. 시각화
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # 유효전력 막대 그래프
    ax1.bar(weekday_avg.index, weekday_avg['activePower'], color='dodgerblue', alpha=0.7, label='Avg ActivePower')
    ax1.set_ylabel('Active Power (Average)', color='dodgerblue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='dodgerblue')
    
    # 무효전력 선 그래프 (이중 축)
    ax2 = ax1.twinx()
    ax2.plot(weekday_avg.index, weekday_avg['reactivePowerLagging'], color='orchid', marker='o', linewidth=3, label='Avg Lagging')
    ax2.set_ylabel('Reactive Power Lagging (Average)', color='orchid', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='orchid')
    
    plt.title('Average Power Load by Weekday (Feb 2025)', fontsize=15)
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.grid(axis='y', alpha=0.3)
    plt.show()
# 
plot_weekday_load_analysis(df)

1. 유효전력(ActivePower)
- 요일에 관계없이 평균 유효전력(ActivePower)이 약 3000 수준으로 거의 일정하게 유지
> 해당 설비가 공장의 기본 운영을 책임지는 기저 부하(Base Load) 설비임을 증명

2. 무효전력(ReactiveRower Lagging)
- 목요일과 일요일에 무효전력 평균이 상대적으로 높게 나타남
> 같은 일을 하더라도 목요일과 일요일에 전력 효율(역률)이 미세하게 떨어진다는 뜻

### 요일별 시간대별 히트맵

In [ ]:
# 요일별/시간대별 유효전력 히트맵
def plot_weekday_hour_load_analysis(df):
    # 1. 요일 정보 추출 (0: 월요일, 6: 일요일)
    df['weekday'] = df['localtime'].dt.weekday
    df['day_name'] = df['localtime'].dt.day_name()
    
    # 요일 순서 정렬을 위한 리스트
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    # 2. 요일별 평균 계산
    weekday_avg = df.groupby('day_name')[['activePower', 'reactivePowerLagging']].mean().reindex(weekday_order)
    df['hour'] = df['localtime'].dt.hour
    pivot_table = df.pivot_table(index='day_name', columns='hour', values='activePower', aggfunc='mean').reindex(weekday_order)

    plt.figure(figsize=(15, 8))
    sns.heatmap(pivot_table, cmap='YlGnBu', annot=False)
    plt.title('Hourly Load Pattern by Weekday')
    plt.show()

plot_weekday_hour_load_analysis(df)

피크 시간
1. 월요일 5시
2. 화요일 2-5시
3. 목요일 11시
4. 토요일 12시
5. 일요일 13시

- 모든 요일의 부하가 균일하지 않고 특정 시간에 '피크'가 발생
- 해당 시간대 제외 다른 시간대에 다른 설비를 가동?

In [ ]:
import matplotlib.pyplot as plt

def plot_daily_power_comparison(df, target_date):
    # 1. 특정 날짜 데이터 필터링
    daily_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()
    daily_df = daily_df.sort_values('localtime')

    if daily_df.empty:
        print(f"{target_date}에 해당하는 데이터가 없습니다.")
        return

    # 2. 시각화 설정
    fig, ax1 = plt.subplots(figsize=(15, 6))
    
    # 왼쪽 축: 유효전력 (ActivePower)
    ax1.plot(daily_df['localtime'], daily_df['activePower'], color='dodgerblue', label='Active Power (P)', alpha=0.8, linewidth=1.5)
    ax1.set_xlabel('Time (24 Hours)', fontsize=12)
    ax1.set_ylabel('Active Power (kW)', color='dodgerblue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='dodgerblue')
    
    # 오른쪽 축: 지연무효전력 (Lagging)
    ax2 = ax1.twinx()
    ax2.plot(daily_df['localtime'], daily_df['reactivePowerLagging'], color='orchid', label='Lagging Reactive (Q)', alpha=0.7, linewidth=1.2)
    ax2.set_ylabel('Lagging Reactive Power (kVar)', color='orchid', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='orchid')

    # 3. 그래프 꾸미기
    plt.title(f'[1(PM-3)] Active vs Lagging Reactive Power ({target_date})', fontsize=15)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # 범례 합치기
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper right')

    plt.tight_layout()
    plt.show()

# 실행 예시 (원하는 날짜 입력)
plot_daily_power_comparison(df, '2025-02-10')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_hourly_detail_analysis(df, target_date, target_hour):
    """
    df: 데이터프레임
    target_date: '2025-02-10' 형태의 날짜
    target_hour: 0~23 사이의 정수 (예: 5 시를 보고 싶다면 5)
    """
    # 1. 시간 범위 설정 (시작 시간부터 1시간 동안)
    start_time = pd.to_datetime(f"{target_date} {target_hour:02d}:00:00")
    end_time = start_time + pd.Timedelta(hours=1)
    
    # 2. 해당 1시간 데이터 필터링
    hourly_df = df[(df['localtime'] >= start_time) & (df['localtime'] < end_time)].copy()
    hourly_df = hourly_df.sort_values('localtime')

    if hourly_df.empty:
        print(f"{start_time} ~ {end_time} 구간에 데이터가 없습니다.")
        return

    # 3. 시각화 (이중 축 활용)
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # Active Power (유효전력) - 막대나 선으로 표현
    ax1.plot(hourly_df['localtime'], hourly_df['activePower'], 
             color='tab:blue', label='Active Power', marker='o', markersize=4, linewidth=2)
    ax1.set_ylabel('Active Power (kW)', color='tab:blue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    
    # Reactive Power Lagging (지연 무효전력)
    ax2 = ax1.twinx()
    ax2.plot(hourly_df['localtime'], hourly_df['reactivePowerLagging'], 
             color='tab:red', label='Lagging Reactive', marker='x', markersize=4, linestyle='--', linewidth=1.5)
    ax2.set_ylabel('Lagging Reactive (kVar)', color='tab:red', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='tab:red')

    # 그래프 서식
    plt.title(f'[1(PM-3)] Hourly Precision Analysis ({start_time} ~ {end_time})', fontsize=14)
    ax1.set_xlabel('Time (Minutes)', fontsize=11)
    ax1.grid(True, which='both', linestyle=':', alpha=0.6)
    
    # 범례 합치기
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    plt.tight_layout()
    plt.show()

# 실행 예시: 2025년 2월 10일 오전 5시부터 6시까지의 변화 분석
plot_hourly_detail_analysis(df, '2025-02-10', 5)

In [ ]:
plot_hourly_detail_analysis(df, '2025-02-10', 12)

In [ ]:
plot_hourly_detail_analysis(df, '2025-02-10', 18)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def analyze_energy_peak_relationship(df, target_date):
    # 1. 특정 날짜 데이터 및 Energy Diff 계산
    day_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()
    day_df['energy_diff'] = day_df['accumActiveEnergy'].diff().fillna(0)
    
    # 2. 시각화 (유효전력 vs 에너지 사용량 변화)
    fig, (ax1, ax3) = plt.subplots(2, 1, figsize=(15, 12))
    
    # [그래프 1] ActivePower와 Energy Diff의 일치성 확인
    ax2 = ax1.twinx()
    lns1 = ax1.plot(day_df['localtime'], day_df['activePower'], color='dodgerblue', label='Active Power (kW)', alpha=0.7)
    lns2 = ax2.plot(day_df['localtime'], day_df['energy_diff'], color='orange', label='Energy Diff', alpha=0.8, linewidth=1.5)
    
    # 가설 지점 표시 (activePower > 4500)
    peak_points = day_df[day_df['activePower'] >= 4500]
    ax1.scatter(peak_points['localtime'], peak_points['activePower'], color='red', s=50, label='Peak (>=4500W)')
    
    ax1.set_title(f"Hypothesis 1: ActivePower Peak vs Energy Diff ({target_date})", fontsize=14)
    ax1.set_ylabel("Active Power (kW)")
    ax2.set_ylabel("Energy Diff")
    
    # 범례 합치기
    lns = lns1 + lns2
    labs = [l.get_label() for l in lns]
    ax1.legend(lns, labs, loc='upper right')

    # [그래프 2] Energy Diff와 Lagging의 상관관계 (산점도)
    # 가설: Energy Diff가 높을수록 Lagging도 높다
    scatter = ax3.scatter(day_df['energy_diff'], day_df['reactivePowerLagging'], 
                          c=day_df['activePower'], cmap='viridis', alpha=0.6)
    ax3.set_title("Hypothesis 2: Energy Diff vs Reactive Power Lagging", fontsize=14)
    ax3.set_xlabel("Energy Diff")
    ax3.set_ylabel("Lagging Reactive Power (kVar)")
    plt.colorbar(scatter, ax=ax3, label='Active Power')
    
    plt.tight_layout()
    plt.show()

# 실행 (데이터프레임과 분석하고 싶은 날짜 입력)
analyze_energy_peak_relationship(df, '2025-02-10')

In [ ]:
run_full_eda(modules[1], df_weekly1)

In [ ]:
run_full_eda(modules[2], df_weekly1)

In [ ]:
run_full_eda(modules[3], df_weekly1)

In [ ]:
# 1. 전류 평균 및 불평형률 계산
df['avg_current'] = df[['currentR', 'currentS', 'currentT']].mean(axis=1)
df['max_current_dev'] = df[['currentR', 'currentS', 'currentT']].sub(df['avg_current'], axis=0).abs().max(axis=1)
df['current_unbalance'] = (df['max_current_dev'] / df['avg_current']) * 100

# 2. 역률(PF) 간의 편차 계산 (상이한 효율 확인)
df['pf_std'] = df[['powerFactorR', 'powerFactorS', 'powerFactorT']].std(axis=1)

# 3. 불평형률이 특정 기준(예: 10%)을 넘는 구간 확인
unbalanced_incidents = df[df['current_unbalance'] > 10]
print(f"전류 불평형 10% 초과 발생 횟수: {len(unbalanced_incidents)}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 2025-02-01 데이터만 필터링
target_date = '2025-02-01'
daily_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()

# 2. 시간대별(1시간 단위) 평균 계산 (패턴을 더 명확히 보기 위함)
hourly_active = daily_df.set_index('localtime')['activePower'].resample('1H').mean()

# 3. 그래프 그리기
plt.figure(figsize=(12, 6))
plt.plot(hourly_active.index, hourly_active.values, marker='o', color='dodgerblue', linewidth=2)

plt.title(f'Hourly Active Power Usage ({target_date})', fontsize=14)
plt.xlabel('Time (Hour)', fontsize=12)
plt.ylabel('Average Active Power (kW)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# X축 시간 포맷 설정
plt.xticks(hourly_active.index[::2], [i.strftime('%H:%M') for i in hourly_active.index[::2]])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 2025-02-01 데이터만 필터링
target_date = '2025-02-02'
daily_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()

# 2. 시간대별(1시간 단위) 평균 계산 (패턴을 더 명확히 보기 위함)
hourly_active = daily_df.set_index('localtime')['activePower'].resample('1H').mean()

# 3. 그래프 그리기
plt.figure(figsize=(12, 6))
plt.plot(hourly_active.index, hourly_active.values, marker='o', color='dodgerblue', linewidth=2)

plt.title(f'Hourly Active Power Usage ({target_date})', fontsize=14)
plt.xlabel('Time (Hour)', fontsize=12)
plt.ylabel('Average Active Power (kW)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# X축 시간 포맷 설정
plt.xticks(hourly_active.index[::2], [i.strftime('%H:%M') for i in hourly_active.index[::2]])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 2025-02-01 데이터만 필터링
target_date = '2025-02-03'
daily_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()

# 2. 시간대별(1시간 단위) 평균 계산 (패턴을 더 명확히 보기 위함)
hourly_active = daily_df.set_index('localtime')['activePower'].resample('1H').mean()

# 3. 그래프 그리기
plt.figure(figsize=(12, 6))
plt.plot(hourly_active.index, hourly_active.values, marker='o', color='dodgerblue', linewidth=2)

plt.title(f'Hourly Active Power Usage ({target_date})', fontsize=14)
plt.xlabel('Time (Hour)', fontsize=12)
plt.ylabel('Average Active Power (kW)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# X축 시간 포맷 설정
plt.xticks(hourly_active.index[::2], [i.strftime('%H:%M') for i in hourly_active.index[::2]])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 2025-02-01 데이터만 필터링
target_date = '2025-02-04'
daily_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()

# 2. 시간대별(1시간 단위) 평균 계산 (패턴을 더 명확히 보기 위함)
hourly_active = daily_df.set_index('localtime')['activePower'].resample('1H').mean()

# 3. 그래프 그리기
plt.figure(figsize=(12, 6))
plt.plot(hourly_active.index, hourly_active.values, marker='o', color='dodgerblue', linewidth=2)

plt.title(f'Hourly Active Power Usage ({target_date})', fontsize=14)
plt.xlabel('Time (Hour)', fontsize=12)
plt.ylabel('Average Active Power (kW)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# X축 시간 포맷 설정
plt.xticks(hourly_active.index[::2], [i.strftime('%H:%M') for i in hourly_active.index[::2]])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 2025-02-01 데이터만 필터링
target_date = '2025-02-05'
daily_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()

# 2. 시간대별(1시간 단위) 평균 계산 (패턴을 더 명확히 보기 위함)
hourly_active = daily_df.set_index('localtime')['activePower'].resample('1H').mean()

# 3. 그래프 그리기
plt.figure(figsize=(12, 6))
plt.plot(hourly_active.index, hourly_active.values, marker='o', color='dodgerblue', linewidth=2)

plt.title(f'Hourly Active Power Usage ({target_date})', fontsize=14)
plt.xlabel('Time (Hour)', fontsize=12)
plt.ylabel('Average Active Power (kW)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# X축 시간 포맷 설정
plt.xticks(hourly_active.index[::2], [i.strftime('%H:%M') for i in hourly_active.index[::2]])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 2025-02-01 데이터만 필터링
target_date = '2025-02-06'
daily_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()

# 2. 시간대별(1시간 단위) 평균 계산 (패턴을 더 명확히 보기 위함)
hourly_active = daily_df.set_index('localtime')['activePower'].resample('1H').mean()

# 3. 그래프 그리기
plt.figure(figsize=(12, 6))
plt.plot(hourly_active.index, hourly_active.values, marker='o', color='dodgerblue', linewidth=2)

plt.title(f'Hourly Active Power Usage ({target_date})', fontsize=14)
plt.xlabel('Time (Hour)', fontsize=12)
plt.ylabel('Average Active Power (kW)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# X축 시간 포맷 설정
plt.xticks(hourly_active.index[::2], [i.strftime('%H:%M') for i in hourly_active.index[::2]])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_smoothed_power(df, target_date):
    day_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()
    
    # 1. 15분 및 1시간 이동 평균 계산
    day_df['rolling_15m'] = day_df['activePower'].rolling(window=15, center=True).mean()
    day_df['rolling_1h'] = day_df['activePower'].rolling(window=60, center=True).mean()
    
    plt.figure(figsize=(15, 6))
    
    # 원본 데이터 (연하게)
    plt.plot(day_df['localtime'], day_df['activePower'], color='lightgray', alpha=0.5, label='Original (Raw)')
    
    # 이동 평균 데이터 (진하게)
    plt.plot(day_df['localtime'], day_df['rolling_15m'], color='orange', label='15-Min Moving Average', linewidth=1.5)
    plt.plot(day_df['localtime'], day_df['rolling_1h'], color='red', label='1-Hour Moving Average', linewidth=2)
    
    plt.title(f"Power Trend Smoothing ({target_date})", fontsize=15)
    plt.legend()
    plt.show()

plot_smoothed_power(df, '2025-02-01')

In [ ]:
plot_smoothed_power(df, '2025-02-02')

In [ ]:
import matplotlib.pyplot as plt

def plot_smoothed_power(df, target_date):
    day_df = df[df['localtime'].dt.date == pd.to_datetime(target_date).date()].copy()
    
    # 1. 15분 및 1시간 이동 평균 계산
    day_df['rolling_5m'] = day_df['activePower'].rolling(window=5, center=True).mean()
    day_df['rolling_15m'] = day_df['activePower'].rolling(window=15, center=True).mean()
    day_df['rolling_1h'] = day_df['activePower'].rolling(window=60, center=True).mean()
    plt.figure(figsize=(15, 6))
    
    # 원본 데이터 (연하게)
    plt.plot(day_df['localtime'], day_df['activePower'], color='lightgray', alpha=0.5, label='Original (Raw)')
    
    # 이동 평균 데이터 (진하게)
    plt.plot(day_df['localtime'], day_df['rolling_5m'], color='orange', label='5-Min Moving Average', linewidth=1.5)
    plt.plot(day_df['localtime'], day_df['rolling_15m'], color='red', label='15-Min Moving Average', linewidth=2)
    plt.plot(day_df['localtime'], day_df['rolling_1h'], color='blue', label='1-Hour Moving Average', linewidth=2)

    plt.title(f"Power Trend Smoothing ({target_date})", fontsize=15)
    plt.legend()
    plt.show()

plot_smoothed_power(df, '2025-02-01')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_weekly_hourly_pattern(df):
    # 1. 요일 정보와 시간 정보 추출
    df['hour'] = df['localtime'].dt.hour
    df['weekday'] = df['localtime'].dt.day_name()
    
    # 요일 순서 정렬
    ordered_days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    
    # 2. 요일별, 시간별 유효전력 평균 계산
    pivot_df = df.groupby(['weekday', 'hour'])['activePower'].mean().unstack(level=0)
    pivot_df = pivot_df[ordered_days] # 요일 순서 맞추기

    # 3. 그래프 그리기
    plt.figure(figsize=(15, 8))
    sns.lineplot(data=df, x='hour', y='activePower', hue='weekday', 
                 hue_order=ordered_days, palette='viridis', ci=None, linewidth=2.5)
    
    plt.title('Weekly Operation Cycle Analysis: Average Active Power by Hour', fontsize=16)
    plt.xlabel('Hour of Day (0-23)', fontsize=12)
    plt.ylabel('Average Active Power (kW)', fontsize=12)
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', linestyle='--', alpha=0.6)
    plt.legend(title='Day of Week', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()

# 실행
plot_weekly_hourly_pattern(df)

In [ ]:
import numpy as np

# 1. 볼린저 밴드 계산 (1시간 윈도우)
window = 60
df['MA'] = df['activePower'].rolling(window=window).mean()
df['std'] = df['activePower'].rolling(window=window).std()

df['Upper'] = df['MA'] + (2 * df['std'])
df['Lower'] = df['MA'] - (2 * df['std'])

# 2. 이상치(상한선 돌파) 데이터만 추출
anomalies = df[df['activePower'] > df['Upper']]

# 3. 시각화
plt.figure(figsize=(15, 7))
plt.plot(df['localtime'], df['activePower'], color='lightgray', label='Active Power', alpha=0.5)
plt.plot(df['localtime'], df['Upper'], color='red', label='Upper Band (Anomaly Threshold)', linestyle='--')
plt.scatter(anomalies['localtime'], anomalies['activePower'], color='crimson', s=10, label='Anomaly Point')

plt.title("Anomaly Detection using Bollinger Bands (95.4% Confidence)")
plt.legend()
plt.show()

print(f"전체 데이터 중 이상치 탐지 비율: {len(anomalies)/len(df)*100:.2f}%")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. 시각화 엔진 설정 (에러 방지용)
plt.rcParams['agg.path.chunksize'] = 10000
plt.rcParams['font.family'] = 'Malgun Gothic' # 한글 깨짐 방지 (윈도우 기준)
plt.rcParams['axes.unicode_minus'] = False

# 2. 데이터 전처리: 15분 단위 리샘플링
# 'localtime'이 이미 datetime 형식이라고 가정합니다.
df_15m = df.set_index('localtime').resample('15T').mean().reset_index()

# 3. 이동평균 및 볼린저 밴드 계산 (15분 데이터 기준)
# 윈도우 4는 15분 * 4 = 1시간을 의미합니다.
window = 4 
df_15m['MA_1h'] = df_15m['activePower'].rolling(window=window).mean()
df_15m['std_1h'] = df_15m['activePower'].rolling(window=window).std()

# 상한선(Upper Band): 평균 + 2*표준편차 (95.4% 신뢰구간)
df_15m['Upper'] = df_15m['MA_1h'] + (2 * df_15m['std_1h'])
df_15m['Lower'] = df_15m['MA_1h'] - (2 * df_15m['std_1h'])

# 4. 이상치(Anomaly) 정의
# 유효전력이 상한선을 돌파하는 지점 추출
anomalies = df_15m[df_15m['activePower'] > df_15m['Upper']]

# 5. 시각화 (전체 기간 또는 특정 기간 슬라이싱)
def plot_advanced_analysis(data, anomaly_data, title="15분 단위 전력 분석 및 이상 탐지"):
    plt.figure(figsize=(16, 8))
    
    # 원본(15분 평균) 데이터
    plt.plot(data['localtime'], data['activePower'], color='lightgray', label='유효전력 (15m Avg)', alpha=0.7)
    
    # 이동평균선 (흐름)
    plt.plot(data['localtime'], data['MA_1h'], color='dodgerblue', label='1시간 이동평균 (Trend)', linewidth=2)
    
    # 볼린저 밴드 상한선 (임계치)
    plt.plot(data['localtime'], data['Upper'], color='orange', linestyle='--', label='상한 임계치 (Upper Band)')
    
    # 이상치 점 찍기
    plt.scatter(anomaly_data['localtime'], anomaly_data['activePower'], color='red', s=20, label='이상 부하 발생(Anomaly)', zorder=3)
    
    plt.title(title, fontsize=15)
    plt.xlabel('시간')
    plt.ylabel('유효전력 (kW)')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()

# 실행 (전체 기간 시각화)
plot_advanced_analysis(df_15m, anomalies)

# 6. 통계적 요약 출력
print(f"--- 분석 결과 요약 (15분 단위) ---")
print(f"전체 관측 구간 수: {len(df_15m)} 개")
print(f"이상 부하(상한선 돌파) 발생 건수: {len(anomalies)} 건")
print(f"전체 시간 대비 이상 발생 비율: {len(anomalies)/len(df_15m)*100:.2f}%")

In [ ]:
modules